In [1]:
import numpy as np
import pandas as pd
import uwb_dataset

data = uwb_dataset.import_from_files()  # numpy array

# column names based on CSV format
base_cols = [
    "NLOS","RANGE","FP_IDX","FP_AMP1","FP_AMP2","FP_AMP3",
    "STDEV_NOISE","CIR_PWR","MAX_NOISE","RXPACC","CH",
    "FRAME_LEN","PREAM_LEN","BITRATE","PRFR"
]
cir_cols = [f"CIR{i}" for i in range(1016)]
cols = base_cols + cir_cols

df = pd.DataFrame(data, columns=cols)
df.head()

../dataset/uwb_dataset_part2.csv
../dataset/uwb_dataset_part3.csv
../dataset/uwb_dataset_part1.csv
../dataset/uwb_dataset_part4.csv
../dataset/uwb_dataset_part5.csv
../dataset/uwb_dataset_part7.csv
../dataset/uwb_dataset_part6.csv


,NLOS,RANGE,FP_IDX,FP_AMP1,FP_AMP2,FP_AMP3,STDEV_NOISE,CIR_PWR,MAX_NOISE,RXPACC,...,CIR1006,CIR1007,CIR1008,CIR1009,CIR1010,CIR1011,CIR1012,CIR1013,CIR1014,CIR1015
0,1.0,6.18,749.0,4889.0,13876.0,10464.0,240.0,9048.0,3668.0,1024.0,...,0.798828,0.916016,0.574219,0.270508,0.709961,0.358398,0.784180,0.799805,0.456055,0.75
1,1.0,4.54,741.0,2474.0,2002.0,1593.0,68.0,6514.0,1031.0,1024.0,...,0.282227,0.222656,0.104492,0.475586,0.479492,0.394531,0.326172,0.205078,0.099609,0.00
2,1.0,4.39,744.0,1934.0,2615.0,4114.0,52.0,2880.0,796.0,1024.0,...,0.120117,0.274414,0.471680,0.094727,0.265625,0.071289,0.122070,0.165039,0.177734,0.00
3,1.0,1.27,748.0,16031.0,17712.0,10420.0,64.0,12855.0,1529.0,323.0,...,0.523220,0.427245,0.678019,0.291022,0.696594,0.479876,0.532508,0.860681,0.984520,0.00
4,0.0,1.16,743.0,20070.0,19886.0,15727.0,76.0,11607.0,2022.0,296.0,...,0.293919,0.145270,1.209459,1.040541,0.445946,0.442568,0.344595,0.425676,0.550676,0.00


Basic Cleaning/Validation

In [8]:
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())

#drop duplicates if any
df = df.drop_duplicates()



Shape: (42000, 1035)
Missing values: 0
Duplicates: 0


feature engineering

In [9]:
# FP “power” aggregate
df["FP_POWER"] = (df["FP_AMP1"]**2 + df["FP_AMP2"]**2 + df["FP_AMP3"]**2)

# Simple CIR summary stats (fast, useful)
cir = df[cir_cols].to_numpy()
df["CIR_MEAN"] = cir.mean(axis=1)
df["CIR_MAX"]  = cir.max(axis=1)
df["CIR_ENERGY"] = (cir**2).mean(axis=1)  # average energy

df[["FP_POWER","CIR_MEAN","CIR_MAX","CIR_ENERGY"]].head()

,FP_POWER,CIR_MEAN,CIR_MAX,CIR_ENERGY
0,3.259410e+08,0.750593,18.063477,1.764866
1,1.266633e+07,0.407109,13.740234,0.942762
2,2.750358e+07,0.369401,7.645508,0.470664
3,6.792843e+08,1.033827,65.857585,17.356314
4,1.045596e+09,1.321797,71.388514,21.166901


Data reduction using PCA
Explained variance means how much of the CIR information is the 30 components keeping

In [10]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Separate target
y = df["NLOS"].astype(int).to_numpy()

# Features excluding raw CIR for now
non_cir_features = [
    "RANGE","FP_IDX","FP_AMP1","FP_AMP2","FP_AMP3",
    "STDEV_NOISE","CIR_PWR","MAX_NOISE","RXPACC","CH",
    "FRAME_LEN","PREAM_LEN","BITRATE","PRFR",
    "FP_POWER","CIR_MEAN","CIR_MAX","CIR_ENERGY"
]

X_non_cir = df[non_cir_features].to_numpy()
X_cir = df[cir_cols].to_numpy()

# Scale CIR before PCA
scaler_cir = StandardScaler()
X_cir_scaled = scaler_cir.fit_transform(X_cir)

# Choose number of PCA components (start with 20–50; you can tune)
pca = PCA(n_components=30, random_state=42)
X_cir_pca = pca.fit_transform(X_cir_scaled)

print("Explained variance (30 comps):", pca.explained_variance_ratio_.sum())

Explained variance (30 comps): 0.5055526064439727


Final training matrix + scaling

In [11]:
# Combine engineered + PCA features
X = np.hstack([X_non_cir, X_cir_pca])

# Scale final feature matrix (good practice for many models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape

(42000, 48)

Feature importance ranking

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_imp = df[non_cir_features].to_numpy()
y = df["NLOS"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=non_cir_features)\
              .sort_values(ascending=False)

print(importances.head(15))

CIR_ENERGY     0.202210
CIR_MAX        0.149047
RXPACC         0.117474
RANGE          0.110567
CIR_MEAN       0.101352
FP_POWER       0.064990
FP_AMP3        0.055960
CIR_PWR        0.049919
MAX_NOISE      0.039925
FP_AMP2        0.037265
FP_AMP1        0.031161
STDEV_NOISE    0.022457
FP_IDX         0.013433
FRAME_LEN      0.003773
PREAM_LEN      0.000467
dtype: float64
